# 15 Statistical testing
Paired block bootstrap (blocks = days) for PR-AUC differences, DeLong for ROC-AUC, McNemar at a fixed alert budget, Friedman + Nemenyi across test days, Holm correction across comparisons.

In [1]:
import sys, pathlib, json, warnings; warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
def table(name):
    p = ROOT / 'reports' / 'tables' / name
    return pd.read_csv(p) if p.exists() else None

In [2]:
t = table('stat_tests_vs_presage.csv'); t.round(4) if t is not None else 'run scripts/final_evaluation.py'

,vs,delta_pr_auc,ci_lo,ci_hi,p_bootstrap,auc_ref,auc_other,p_delong,mcnemar_p,p_bootstrap_holm
0,presage:concat,-0.0625,-0.0928,-0.0374,0.0025,0.9858,0.9999,0.0000,0.0107,0.02
1,xgboost_tabular+graph_full,-0.0578,-0.0874,-0.0341,0.0025,0.9858,0.9998,0.0000,0.0322,0.02
2,catboost_tabular+graph_full,-0.0564,-0.0893,-0.0306,0.0025,0.9858,0.9998,0.0000,0.0451,0.02
3,presage:no_beh,-0.0404,-0.0587,-0.0225,0.0025,0.9858,0.9959,0.0001,0.0568,0.02
4,graph_hgt,-0.0399,-0.0627,-0.0148,0.0025,0.9858,0.9990,0.0001,0.1634,0.02
5,catboost_tabular+graph_struct,-0.0349,-0.0618,-0.0100,0.0050,0.9858,0.9988,0.0001,0.4390,0.02
6,catboost_all_tabular,-0.0320,-0.0628,-0.0079,0.0050,0.9858,0.9993,0.0001,0.5139,0.02
7,random_forest_tabular+graph_full,-0.0319,-0.0604,-0.0041,0.0300,0.9858,0.9995,0.0001,0.2245,0.03


In [3]:
import json; p = ROOT/'reports/tables/friedman_nemenyi.json'; print(json.dumps(json.load(open(p)), indent=1) if p.exists() else '')

{
 "p": 9.465029418246316e-05,
 "stat": 25.86805555555554,
 "mean_rank": {
  "presage:full": 4.896,
  "presage:concat": 2.708,
  "xgboost_tabular+graph_full": 2.917,
  "catboost_tabular+graph_full": 3.167,
  "presage:no_beh": 3.375,
  "graph_hgt": 3.938
 },
 "critical_difference": 1.539175915871867,
 "n_blocks": 24
}


Interpretation guide: `delta_pr_auc` is PRESAGE minus the comparator; a CI containing 0 (or Holm-adjusted p above 0.05) means the difference is not distinguishable from noise with this test set (about 300 positives). Statistical significance is not the same as operational value; see the cost analysis in notebook 16.